# EO Specimen label parsing

In [6]:
import re
import pandas as pd

# load specimen records associated with EOs as entered into Biotics
Biotics_specimens = pd.read_csv("EO_Specimens_20260130.csv", encoding='latin1')

# search patterns
# word starts with collection, then space or not, # sign, space or not, then any alphabetic characters or digits separated by - or . The second group allows for comma separated number, but requires digits in the second to avoid adding words endlessly.

collection_pattern = re.compile(
    r"(?:\bcollection\s*#\s*)(\w*[\.\-]?\w*[\.\-]?\w*[\.\-]?\w*)(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*",
    re.IGNORECASE
)

catalog_pattern = re.compile(
    r"(?:\bcatalog\s*#\s*)(\w*[\.\-]?\w*[\.\-]?\w*[\.\-]?\w*)(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*",
    re.IGNORECASE
)

accession_pattern = re.compile(
    r"(?:\baccession\s*#\s*)(\w*[\.\-]?\w*[\.\-]?\w*[\.\-]?\w*)(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*(?:[\s,and#\&])*([a-zA-Z]*\d+[\.\-\d]*[a-zA-Z]*)*",
    re.IGNORECASE
)

#O'Kane, S. 1985. Specimen (collection #2103) at University of Colorado Herbarium.
collector_pattern = re.compile(r"^[A-Za-z,.\s]+")
year_pattern = re.compile(r"\b((?:18|19|20)\d{2})\b")

herbarium_df = pd.read_excel('HerbariumCodes.xlsx')

# Create a list of herbarium names and codes
herbarium_dict = dict(zip(herbarium_df['textName'].astype(str), herbarium_df['institutionCode_CNHP']))

# Compile a regular expression pattern for each herbarium name
herbarium_patterns = {name: re.compile(r'\b' + name + r'.*', re.IGNORECASE) for name in herbarium_dict}

# Function to search for herbarium codes in text
def find_herbarium_code(text, patterns, herbarium_dict):
    for name, pattern in patterns.items():
        if pattern.search(text):
            return herbarium_dict[name]
    return None  # If no match is found

# check for collection number ranges
def parse_collection_range(entry):
    if '-' in entry:
        # Check if it's a range (e.g., 475-477)
        if len(entry.split('-')) == 2:
            start, end = entry.split('-')
            try:
                start_num = int(start)
                end_num = int(end)
                # Only expand if the range difference is 4 or less and over 100
                if abs(end_num - start_num) <= 4 and start_num > 100:
                    return list(range(start_num, end_num + 1))
                else:
                    return [entry]
            except ValueError:
                return [entry]
        else:
            return [entry]
    else:
        return [entry]

results = []
for index, row in Biotics_specimens.iterrows():
    text = row["SPECIMEN_DESC"]
    eo_id = row["EO_ID"]
    if year_pattern.search(text) is not None:
        year = year_pattern.search(text).group()
    herbarium = find_herbarium_code(text, herbarium_patterns, herbarium_dict)
    if collector_pattern.search(text) is not None:
        collector = collector_pattern.search(text).group()
    if collection_pattern.search(text) is not None:
        for num in collection_pattern.search(text).groups():
            if num is not None:
                # append the undivided collection number in case it is not really a range
                results.append([eo_id, text, collector, year, herbarium, num, None, None])
                #print([eo_id, collector, year, herbarium, num, None, None])
                if len(parse_collection_range(num))>1:
                    for item in parse_collection_range(num):
                        results.append([eo_id, text, collector, year, herbarium, item, None, None])
    if catalog_pattern.search(text) is not None:
        for num in catalog_pattern.search(text).groups():
            if num is not None:
                # append the undivided collection number in case it is not really a range
                results.append([eo_id, text, collector, year, herbarium, None, num, None])
                if len(parse_collection_range(num))>1:
                    for item in parse_collection_range(num):
                        results.append([eo_id, text, collector, year, herbarium, None, item, None])
    if accession_pattern.search(text) is not None:
        for num in accession_pattern.search(text).groups():
            if num is not None:
                # append the undivided collection number in case it is not really a range
                results.append([eo_id, text, collector, year, herbarium, None, None, num])
                if len(parse_collection_range(num))>1:
                    for item in parse_collection_range(num):
                        results.append([eo_id, text, collector, year, herbarium, None, None, item])

column_names = ["EO_ID", "sourceText", "collector", "year", "institutionCode", "collectionNumber", "catalogNumber", "accessionNumber"]
result_df = pd.DataFrame(results, columns=column_names)

# merge parsed specimen labels with EO SNAME data
EO_df = pd.read_csv("EO_download_20260130.csv", encoding = 'latin1')

result_df = result_df.merge(EO_df, how="left", on="EO_ID")
result_df = result_df[["EO_ID", "SNAME", "ELCODE", "sourceText", "collector", "year", "institutionCode", "collectionNumber", "catalogNumber", "accessionNumber", "PRECISN", "REP_ACC"]]
# filter to only vascular plant specimen labels
result_df = result_df[result_df["ELCODE"].str.startswith('P', na=False)]
result_df.to_csv("EO_Specimens_parsed_20260130.csv", index=False)


# EO specimen label matching to SEINet

In [13]:
import pandas as pd
import re
import numpy as np
# use combination of collector, year, and collection number to identify specimens in Biotics (return those with different SNAMEs to be reviewed manually for updates)
# include two columns in SEINet spreadsheet, one for Matched: Y, S, H; (yes, species discrepancy, herbarium discrepancy) and one for EOID 

EO_specimens = pd.read_csv("EO_Specimens_parsed_20260130.csv", encoding='latin1')
#EO_specimens = pd.read_csv("EO_parse_test.csv")

#from rpy2.robjects import pandas2ri, conversion, default_converter
#from rpy2.robjects.packages import importr

SEINet = pd.read_csv("SEINet_translated_20260202.csv", encoding='latin1')

# Add a new column to SEINet to track matched rows
SEINet["matched"] = None
SEINet["EO_ID_match"] = None
# Define the function to check matches and update SEINet

SEINet = SEINet.rename(columns={'id': 'SEINet_id', 'recordNumber': 'collectionNumber', "otherCatalogNumbers": "accessionNumber", "recordedBy": "collector", "acceptedName": "SNAME"})

number_columns = ['collectionNumber', 'catalogNumber', 'accessionNumber']
# strip leading zeros from SEINet number columns to match the EO labels in case either one got zeros removed also remove ".0" from those that were stored as floats.
for column in number_columns:
    SEINet[column]=SEINet[column].astype(str).str.lstrip('0')
    SEINet[column] = SEINet[column].str.replace(r'\.0$', '', regex=True)
    SEINet[column] = SEINet[column].replace("nan", np.nan)
    EO_specimens[column]=EO_specimens[column].astype(str).str.lstrip('0')
    EO_specimens[column] = EO_specimens[column].str.replace(r'\.0$', '', regex=True)
    EO_specimens[column] = EO_specimens[column].replace("nan", np.nan)

EO_specimens["SEINet_id"]=None
EO_specimens["SEINet_links"] = None
EO_specimens["SEINet_links_all"] = None
EO_specimens["match_status"]=None
EO_specimens["year_check"]=None
EO_specimens["collector_check"]=None

for index, row in EO_specimens.iterrows():
    # Step 1: Filter SEINet by year column
    matched_rows = SEINet[SEINet["year"]==row["year"]]
    if not matched_rows.empty:
        year_check="Y"
    else:
        print("fail")
        print(row["year"])
        year_check="N"
    # take the last name from the collectors
    last_name = re.split(r'[,.\s]+', row['collector'])[0].strip()
    matched_rows = matched_rows[matched_rows['collector'].str.contains(last_name, case=False, na=False)]
    SEINet_ids=None
    SEINet_ids_all=None
    SEINet_links=None
    SEINet_links_all=None
    match_status="N"
    if not matched_rows.empty:
        collector_check="Y"
    else:
        collector_check="N"
        collector_check=f"Lastname: {last_name}, Collector: {row['collector']}"
        EO_specimens.at[index, "SEINet_id"] = SEINet_ids
        EO_specimens.at[index, "match_status"] = match_status
        EO_specimens.at[index, "year_check"] = year_check
        EO_specimens.at[index, "collector_check"] = collector_check

    # Step 2: Identify the first non-NaN column in number_columns for the row
    non_na_col = next((col for col in number_columns if not pd.isna(row[col])), None)
    # Step 3: If a non-NaN number_columns exists, further filter matched_rows by this column
    if non_na_col:
        #print(row['SName'])
        #print(row[non_na_col])
        #print(matched_rows[non_na_col])
        value = row[non_na_col]
        # allow for numbers incorrectly attributed in Biotics, ex: a collectionNumber called a catalogNumber
        matched_rows = matched_rows[(matched_rows['collectionNumber'] == value) | (matched_rows['catalogNumber'] == value) | (matched_rows['accessionNumber'] == value)]   
        # if there are any matches to year, collector last name, and collection/catalog/accession number
        if not matched_rows.empty:
            # mark those who made it to the species check
            SEINet.loc[matched_rows.index, "matched"] = "S"
            # update matching ids
            SEINet_ids_all = matched_rows["SEINet_id"]
            SEINet_links_all = matched_rows["references"]
            if len(SEINet_ids_all) == 1:
                SEINet_ids_all = SEINet_ids_all.iloc[0]  # Extract the single value
            else:
                SEINet_ids_all = "; ".join(SEINet_ids_all.astype(str))  # Join multiple IDs as a string
            if len(SEINet_links_all) == 1:
                SEINet_links_all = SEINet_links_all.iloc[0]  # Extract the single value
            else:
                SEINet_links_all = "; ".join(SEINet_links_all.astype(str))  # Join multiple IDs as a string
            match_status = "S"
            # recording matching EO_ID or document overwrites
            if SEINet.loc[matched_rows.index, "EO_ID_match"] is not None:
                SEINet.loc[matched_rows.index, "EO_ID_match"] = row["EO_ID"]
            else:
                print("Caution: overwriting!")
                print(SEINet.loc[matched_rows.index])
            # check species match
            matched_rows = matched_rows[matched_rows["SNAME"] == row["SNAME"]]
            if not matched_rows.empty:
                # mark those who made it to the herbarium check
                SEINet.loc[matched_rows.index, "matched"] = "H"
                # update matching ids
                SEINet_ids = matched_rows["SEINet_id"]
                if len(SEINet_ids) == 1:
                    SEINet_ids = SEINet_ids.iloc[0]  # Extract the single value
                else:
                    SEINet_ids = "; ".join(SEINet_ids.astype(str))  # Join multiple IDs as a string
                match_status = "H"
                matched_rows = matched_rows[matched_rows["institutionCode"] == row["institutionCode"]]
                if not matched_rows.empty:
                    # mark those complete matches
                    SEINet.loc[matched_rows.index, "matched"] = "Y"
                    # update matching ids
                    SEINet_ids = matched_rows["SEINet_id"]
                    SEINet_links = matched_rows["references"]
                    if len(SEINet_ids) == 1:
                        SEINet_ids = SEINet_ids.iloc[0]  # Extract the single value
                    else:
                        SEINet_ids = "; ".join(SEINet_ids.astype(str))  # Join multiple IDs as a string
                    if len(SEINet_links) == 1:
                        SEINet_links = SEINet_links.iloc[0]  # Extract the single value
                    else:
                        SEINet_links = "; ".join(SEINet_links.astype(str))  # Join multiple IDs as a string
                    match_status = "Y"
            EO_specimens.at[index, "SEINet_id"] = SEINet_ids
            EO_specimens.at[index, "SEINet_id_all"] = SEINet_ids_all
            EO_specimens.at[index, "SEINet_links"] = SEINet_links
            EO_specimens.at[index, "SEINet_links_all"] = SEINet_links_all
            EO_specimens.at[index, "match_status"] = match_status
            EO_specimens.at[index, "year_check"] = year_check
            EO_specimens.at[index, "collector_check"] = collector_check
        else:
            EO_specimens.at[index, "SEINet_id"] = SEINet_ids
            EO_specimens.at[index, "SEINet_id_all"] = SEINet_ids_all
            EO_specimens.at[index, "SEINet_links"] = SEINet_links
            EO_specimens.at[index, "SEINet_links_all"] = SEINet_links_all
            EO_specimens.at[index, "match_status"] = match_status
            EO_specimens.at[index, "year_check"] = year_check
            EO_specimens.at[index, "collector_check"] = collector_check
    else: 
        EO_specimens.at[index, "SEINet_id"] = None
        EO_specimens.at[index, "SEINet_id_all"] = SEINet_ids_all
        EO_specimens.at[index, "SEINet_links"] = SEINet_links
        EO_specimens.at[index, "SEINet_links_all"] = SEINet_links_all
        EO_specimens.at[index, "match_status"] = "Specimen number missing!"
        EO_specimens.at[index, "year_check"] = year_check
        EO_specimens.at[index, "collector_check"] = collector_check
    
    

EO_specimens.to_csv("EO_specimens_parse_check_20260202.csv", index=False)
#EO_specimens.to_csv("EO_specimens_parse_check_test.csv", index=False)
# process for removing duplicates in EO specimen labels where there were multiple collection number, so we know specimen labels that truly had no match
# Define valid match statuses
valid_statuses = {'Y', 'S', 'H'}
# Create a mask 
mask = EO_specimens.groupby('sourceText')['match_status'].transform(
    lambda x: any(status in valid_statuses for status in x)
) & (EO_specimens['match_status'] == 'N')

# Drop rows where the mask is True
EO_specimens_cleaned = EO_specimens[~mask]
EO_specimens_cleaned.to_csv("EO_specimens_parse_check_20260202_cleaned.csv", index=False)
#EO_specimens_cleaned.to_csv("EO_specimens_parse_check_clean_test.csv", index=False)
SEINet.to_csv("SEINet_specimen_match_20260202.csv", index=False)
#SEINet.to_csv("SEINet_specimen_match_test.csv", index=False)

C:\Users\chollenb\AppData\Local\Temp\ipykernel_33996\430126372.py:13: DtypeWarning: Columns (29,31,36,49,50,51,52,53,54,57,59,60,62,73,77) have mixed types. Specify dtype option on import or set low_memory=False.
  SEINet = pd.read_csv("SEINet_translated_20260202.csv", encoding='latin1')


fail
1856
fail
2050


In [15]:
# filter matched specimen table to include single result per EO, in order of H, S, Y for matches.

# Your custom sort order
sort_order = ['Y', 'H', 'S']

df = pd.read_csv("EO_specimens_parse_check_20260202_cleaned.csv")
# Create a categorical column with custom order
df['match_status_cat'] = pd.Categorical(
    df['match_status'],
    categories=sort_order + sorted(set(df['match_status'].unique()) - set(sort_order)),
    ordered=True
)

# Sort by the categorical column
df_sorted = df.sort_values('match_status_cat').drop(columns='match_status_cat')

df_sorted = df_sorted.drop_duplicates(subset=["EO_ID", "sourceText"], keep="first")

df_sorted.to_excel("EO_specimens_parse_check_20260202_cleanest.xlsx", index=False)

In [6]:
# filter to specimens with match_status = Y or H (either all fields match or just herbarium is wrong)
import pandas as pd

df = pd.read_excel("EO_specimens_parse_check_20260202_cleanest.xlsx")
df = df[df["match_status"].isin(["Y", "H"])]

# filter out species that are sensitive on SEINet
SEINet_obscure = pd.read_excel("SensitiveSpeciesSEINet.xlsx")
SEINet_obscure = SEINet_obscure[SEINet_obscure["ProtectedNameFull"].str.split().str.len() > 1]
SEINet_obscure["ProtectedName"] = SEINet_obscure["ProtectedNameFull"].apply(lambda x: ' '.join(x.split()[:2]))
df["SNAME"] = df["SNAME"].astype(str)
df["SNAME_bin"] = df["SNAME"].apply(lambda x: ' '.join(x.split()[:2]))
# slice to only those that are not protected
df = df[~df["SNAME_bin"].isin(SEINet_obscure["ProtectedName"])]



In [7]:
# filter to H Ranked EOs only (need to merge with EO_Rank data)
EO_df = pd.read_csv("EO_download_20260130.csv", encoding = 'latin1')
df['EO_ID'] = pd.to_numeric(df['EO_ID'], errors='coerce')
EO_df['EO_ID'] = pd.to_numeric(EO_df['EO_ID'], errors='coerce')

df = df.merge(EO_df[["EO_ID", "EORANK", "RND_GRNK", "SRANK", "SCOMNAME"]], how = "left", on="EO_ID")

df = df.drop_duplicates(subset=["EO_ID", "sourceText"], keep="first")

print("Columns in df:", df.columns.tolist())
df = df[df["EORANK"]=="H"]

# merge EO_specimens with SEINet layer on SEINetID (may need to split on semicolon)
df = df.assign(SEINet_id=df['SEINet_id'].str.split(';')).explode('SEINet_id')
df['SEINet_id'] = df['SEINet_id'].str.strip()
print("Columns in df part 2:", df.columns.tolist())

SEINet = pd.read_csv("SEINet_specimen_match_20260202.csv", encoding='latin1')
df['SEINet_id'] = pd.to_numeric(df['SEINet_id'], errors='coerce')
SEINet['SEINet_id'] = pd.to_numeric(SEINet['SEINet_id'], errors='coerce')

df = df.merge(SEINet, how = "left", on="SEINet_id", suffixes=["", "_SEINet"])

df.to_csv("SEINet_historic_EO_matched_20260202.csv", index=False)





Columns in df: ['EO_ID', 'SNAME', 'ELCODE', 'sourceText', 'collector', 'year', 'institutionCode', 'collectionNumber', 'catalogNumber', 'accessionNumber', 'PRECISN', 'REP_ACC', 'SEINet_id', 'SEINet_links', 'SEINet_links_all', 'match_status', 'year_check', 'collector_check', 'SEINet_id_all', 'SNAME_bin', 'EORANK', 'RND_GRNK', 'SRANK', 'SCOMNAME']
Columns in df part 2: ['EO_ID', 'SNAME', 'ELCODE', 'sourceText', 'collector', 'year', 'institutionCode', 'collectionNumber', 'catalogNumber', 'accessionNumber', 'PRECISN', 'REP_ACC', 'SEINet_id', 'SEINet_links', 'SEINet_links_all', 'match_status', 'year_check', 'collector_check', 'SEINet_id_all', 'SNAME_bin', 'EORANK', 'RND_GRNK', 'SRANK', 'SCOMNAME']


C:\Users\chollenb\AppData\Local\Temp\ipykernel_23052\2364328969.py:18: DtypeWarning: Columns (29,31,36,49,50,51,52,53,54,57,59,60,62,73,77) have mixed types. Specify dtype option on import or set low_memory=False.
  SEINet = pd.read_csv("SEINet_specimen_match_20260202.csv", encoding='latin1')


In [13]:
import pandas as pd

df = pd.read_csv("SEINet_historic_EO_matched_20260202.csv")
EO_df = pd.read_csv("EO_download_20260130.csv", encoding='latin1')

# select required fields

df = df.merge(EO_df[["EO_ID", "FIRSTOBS", "LASTOBS"]])

print(df.columns.to_list())
df = df.rename(columns={
    'EO_ID': 'OBS_ID', 'SNAME': 'SPECIES_NAME', 'SCOMNAME': 'COMMON_NAME', 'EORANK': 'OBS_RANK', 'year': "YEAR",
    'sourceText': 'SPECIMEN_DESC', 'RND_GRNK': 'GLOBAL_RARITY', 'SRANK': 'STATE_RARITY', 'SEINet_links_all': 'All_URLs'
})
print(f"#2: {df.columns.to_list()}")


new_fields = {
    "Organization__if_applicable_": "",
    "Visited_By": "",
    "Visit_Date__YYYY_MM_DD_": "",
    "iNat_Username": "",
    "Found_": "",
    "Total_Hours": float(0),
    "Notes": ""
}

df = df.assign(**new_fields)

ordered_cols = [
    "Organization__if_applicable_",
    "Visited_By",
    "Visit_Date__YYYY_MM_DD_",
    "iNat_Username",
    "Found_",
    "Total_Hours",
    "Notes",
    'OBS_ID',
    'YEAR',
    'SPECIES_NAME',
    'COMMON_NAME',
    'SPECIMEN_DESC',
    'OBS_RANK',
    'GLOBAL_RARITY',
    'STATE_RARITY',
    'FIRSTOBS',
    'LASTOBS',
    'SEINet_id',
    'SEINet_links',
    'All_URLs',
    'occurrenceRemarks',
    'habitat',
    'substrate',
    'locality',
    'locationRemarks',
    'decimalLatitude',
    'decimalLongitude',
    'coordinateUncertaintyInMeters',
    'verbatimElevation'
]

output_df = df[ordered_cols]

# Add previously filled out records from old volunteer AGOL

old_df = pd.read_excel("Outreach_layer_old_export_20260203.xls")
old_df = old_df.drop(columns = ["OBJECTID", "GlobalID"])
old_df = old_df[(old_df["Organization__if_applicable_"].notna()) | (old_df["Visited_By"].notna())]

# ensure these old volunteer records are not duplicated
output_df = output_df[~(output_df["OBS_ID"].isin(old_df["OBS_ID"]))]

output_df = pd.concat([output_df, old_df], ignore_index=True)

output_df.to_csv("Volunteer_update_20260203.csv", index=False)

# Use ArcGIS to filter to only USFS and BLM land with access==Y and add domain for Found?



['EO_ID', 'SNAME', 'ELCODE', 'sourceText', 'collector', 'year', 'institutionCode', 'collectionNumber', 'catalogNumber', 'accessionNumber', 'PRECISN', 'REP_ACC', 'SEINet_id', 'SEINet_links', 'SEINet_links_all', 'match_status', 'year_check', 'collector_check', 'SEINet_id_all', 'SNAME_bin', 'EORANK', 'RND_GRNK', 'SRANK', 'SCOMNAME', 'institutionCode_SEINet', 'collectionCode', 'ownerInstitutionCode', 'basisOfRecord', 'occurrenceID', 'catalogNumber_SEINet', 'accessionNumber_SEINet', 'higherClassification', 'kingdom', 'phylum', 'class', 'order', 'family', 'scientificName', 'taxonID', 'scientificNameAuthorship', 'genus', 'subgenus', 'specificEpithet', 'verbatimTaxonRank', 'infraspecificEpithet', 'cultivarEpithet', 'tradeName', 'taxonRank', 'identifiedBy', 'dateIdentified', 'identificationReferences', 'identificationRemarks', 'taxonRemarks', 'identificationQualifier', 'typeStatus', 'collector_SEINet', 'associatedCollectors', 'collectionNumber_SEINet', 'eventDate', 'eventDate2', 'year_SEINet', 

C:\Users\chollenb\AppData\Local\Temp\ipykernel_24488\2393690619.py:3: DtypeWarning: Columns (3,10,11,12,13,14,16,20,21,22,23,24,25,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  SEINet = pd.read_csv(r"C:\Users\chollenb\OneDrive - Colostate\Documents\Data\SEINet\SEINet_translated_01152025.csv")
